# PRJEB30386 — эталон после аннотации → PCR1 → фрагментация → PCR2 → объединённая модель на основе UMI-консенсуса PE150

Биологический эталон симуляции строится только из `post_annotation_filtered/airr_pass`. Эмпирическая модель ошибок секвенирования обучается независимо на исходных библиотеках: исходной AIRR-аннотации и ридах `pr_trimmed`, с консенсусами UMI-семейств и отложенными ридами. Поэтому последующая фильтрация аннотаций не делает профиль ошибок искусственно чистым.

**Фрагментация:** `uniform single random-cut`.


## Выходные данные

Имя ветки собирается в ячейке параметров из базового имени, режима эталона, `LIBRARY_INPUT_SCALE` и `READ_BUDGET_MULTIPLIER`, например:

`results/PRJEB30386/simulated/<base>_amp_umimode_in10x_rb3x/`

Каталог `qc/` предназначен для контроля качества этой симуляции.


## 1. Окружение

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math, re, shutil, json, hashlib
from pathlib import Path
from collections import defaultdict
import numpy as np

_ENV_CANDIDATES = [
    os.environ.get("BCR_ENV", ""),
    os.environ.get("CONDA_PREFIX", ""),
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
    "/Users/epishkin/mamba/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if p and os.path.isdir(str(Path(p) / "bin"))), None)
if not _CONDA_ENV:
    raise FileNotFoundError("bcr_env not found; activate it or set BCR_ENV")

os.environ["PATH"] = str(Path(_CONDA_ENV) / "bin") + ":" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in [
    str(Path(_CONDA_ENV) / "lib/python3.11/site-packages"),
    str(Path(_CONDA_ENV) / "lib/python3.12/site-packages"),
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

for tool in ("iss", "bowtie2", "bowtie2-build", "samtools", "igblastn"):
    p = shutil.which(tool)
    if not p:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {p}")
print("numpy:", np.__version__)


## 2. Параметры

In [ ]:
BRANCH_NAME = "insilicoseq_150bp_custom_umi_consensus_post_annotation_filtered_ultrasonic_like"

# ---------------------------------------------------------------------
# Эталон, входная порция библиотеки и глубина (входят в имя ветки)
# ---------------------------------------------------------------------
# amplicon_with_primers: физический ампликон PCR1 = V-leader праймер + склеенный
#   рид (остаток leader, V…J, C-фрагмент) + revcomp(C-праймер). Праймеры вырезаны
#   на стадии pr_trimmed, их имена восстанавливаются из заголовков pr_trimmed.
# vj_only: эталон обрезан ровно до V…J.
TRUTH_TEMPLATE_MODE = "amplicon_with_primers"
# umi_mode: каждый UMI отдаёт одну исходную копию самому частому варианту V…J
#   среди своих ридов; варианты без собственных UMI (ошибки) исключаются.
# none: UMI засчитывается каждому варианту, где встретился.
UMI_COLLAPSE_MODE = "umi_mode"
# Число молекул PCR1-пула, идущих на фрагментацию, относительно числа исходных
# UMI-молекул. 1.0 = стресс-тест (копии одного клона почти не перекрываются);
# в реальной подготовке библиотеки берут на порядки больше копий.
LIBRARY_INPUT_SCALE = 10.0
# Множитель глубины относительно суммы сырых пар исходных библиотек: исходный
# MiSeq читал неразрезанные ампликоны, после фрагментации на молекулу нужно
# несколько пар.
READ_BUDGET_MULTIPLIER = 3.0

# Мягкий size-selection после фрагментации: центр распределения длин вставок (нт).
# Входит в имя ветки. 250 нт для ~430–500-нт ампликона даёт перекрывающиеся 5′/3′-куски
# и заметную долю пар с несеквенированным промежутком между R1 и R2 (mate-linked).
SIZE_SELECTION_TARGET = 250

_TRUTH_TAG = {"amplicon_with_primers": "amp", "vj_only": "vj"}[TRUTH_TEMPLATE_MODE]
if UMI_COLLAPSE_MODE == "umi_mode":
    _TRUTH_TAG += "_umimode"
BRANCH_NAME = f"{BRANCH_NAME}_{_TRUTH_TAG}_in{LIBRARY_INPUT_SCALE:g}x_rb{READ_BUDGET_MULTIPLIER:g}x_ss{SIZE_SELECTION_TARGET}"
SOURCE_STAGE = "post_annotation_filtered"

VOLUME = Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin"))
if not (VOLUME / "results/PRJEB30386").exists():
    found = None
    for start in (Path.cwd().resolve(), Path("/Users/epishkin/workspace/bcr-assembler")):
        for candidate in (start, *start.parents):
            if (candidate / "results/PRJEB30386").exists() and (candidate / "scripts").is_dir():
                found = candidate
                break
        if found:
            break
    if not found:
        raise FileNotFoundError("Cannot locate bcr-assembler root; set BCR_VOLUME")
    VOLUME = found

DATASET = "PRJEB30386"
SOURCE_RUNS = ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"]
RUN_LOCUS = {
    "ERR3004229": "IGH",
    "ERR3004230": "IGH",
    "ERR3004231": "IGK",
    "ERR3004232": "IGL",
}
RUN_LIBRARY = {
    "ERR3004229": "IgM",
    "ERR3004230": "IgG",
    "ERR3004231": "IgK",
    "ERR3004232": "IgL",
}
MIXED_SAMPLE = "PRJEB30386_all_chains"
SAMPLES = [MIXED_SAMPLE]

DATASET_DIR = VOLUME / "results" / DATASET
POST_FILTER_DIR = DATASET_DIR / SOURCE_STAGE
TRUTH_AIRR_DIR = POST_FILTER_DIR / "airr_pass"
FILTERED_FASTQ_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"

RAW_FASTQ_DIR = VOLUME / "raw" / DATASET
PR_FASTQ_DIR = DATASET_DIR / "pr_trimmed" / "fastq"
ORIGINAL_AIRR_DIR = DATASET_DIR / "annotation" / "igblast"

if not FILTER_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Post-annotation filter summary not found: {FILTER_SUMMARY_PATH}. "
        "Run filter_human_post_annotation.ipynb first."
    )
FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())
FILTER_RULES = FILTER_SUMMARY["filters"]

assert FILTER_SUMMARY.get("dataset") == DATASET
assert FILTER_RULES["require_expected_locus"] == RUN_LOCUS
MIN_TEMPLATE_LENGTH = int(FILTER_RULES["min_vj_span_nt"])
MIN_V_IDENTITY = float(FILTER_RULES["min_v_identity_percent"])
MIN_J_IDENTITY = float(FILTER_RULES["min_j_identity_percent"])
MAX_V_SUPPORT = float(FILTER_RULES["max_v_support_evalue"])
MAX_J_SUPPORT = float(FILTER_RULES["max_j_support_evalue"])

TARGET_READ_LENGTH = 150

OUT_BASE = DATASET_DIR / "simulated" / BRANCH_NAME
TRUTH_DIR = OUT_BASE / "00_primary_truth"
PCR1_DIR = OUT_BASE / "01_pcr1"
FRAGMENTATION_DIR = OUT_BASE / "02_fragmentation"
PCR2_DIR = OUT_BASE / "03_pcr2"
ALLOCATION_DIR = OUT_BASE / "04_read_allocation"
FASTQ_NATIVE_DIR = OUT_BASE / "05_fastq_native"
FASTQ_DIR = OUT_BASE / "06_fastq_pe150"
MODEL_DIR = OUT_BASE / "model"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
TEMPLATES_DIR = TRUTH_DIR
COUNTS_DIR = ALLOCATION_DIR

for d in (
    TRUTH_DIR, PCR1_DIR, FRAGMENTATION_DIR, PCR2_DIR, ALLOCATION_DIR,
    FASTQ_NATIVE_DIR, FASTQ_DIR, MODEL_DIR, LOGS_DIR, QC_DIR,
):
    d.mkdir(parents=True, exist_ok=True)

NPROC = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False
COMPRESS = True

# Дополнительные требования симуляции к эталонным данным после
# основной фильтрации аннотаций.
DROP_SEQUENCES_WITH_N = True
REQUIRE_VALID_20NT_UMI = True
STARTING_COPIES_MODE = "unique_umis"

# PCR1: стохастическое ветвление на каждом цикле.
PCR_CYCLES = 25
PCR_EFFICIENCY_MEAN = 0.85
PCR_EFFICIENCY_CONCENTRATION = 80.0
PCR_MAX_COPIES = 10**12

# Фрагментация и отбор библиотеки.
#
# Из PCR1-пула берётся конечная порция молекул, каждая молекула физически
# разрезается; до size-selection проверяется сохранение нуклеотидной массы.
SEQUENCE_TYPE = "amplicon"
FRAGMENTATION_MODEL = "ultrasonic_like_recursive_shearing"
# LIBRARY_INPUT_SCALE задан в начале ячейки и входит в имя ветки.

# SIZE_SELECTION_TARGET задан в начале ячейки и входит в имя ветки.
SIZE_SELECTION_SD = 40
# ISS 2.0.1 simulate_read: assert read_length < len(template); custom model read_length = 150 nt.
MIN_SEQUENCABLE_FRAGMENT_LENGTH = 151

# Flux-Simulator-inspired mechanical/nebulisation abstraction.
# These are tunable simulation parameters, not physical constants.
ULTRASONIC_LIMITING_LENGTH = 200.0
ULTRASONIC_BREAKPOINT_SD_FRACTION = 0.20
ULTRASONIC_MAX_RECURSION_DEPTH = 6

FRAGMENTATION_MODEL_PROVENANCE = {
    "model": FRAGMENTATION_MODEL,
    "library_input": "finite aliquot sampled from PCR1 pool proportional to PCR1 abundance",
    "library_input_scale_vs_starting_copies": LIBRARY_INPUT_SCALE,
    "break_probability": "exp(-limiting_length/current_fragment_length)",
    "limiting_length_nt": ULTRASONIC_LIMITING_LENGTH,
    "breakpoint_distribution": "truncated Gaussian around fragment midpoint",
    "breakpoint_sd_fraction": ULTRASONIC_BREAKPOINT_SD_FRACTION,
    "max_recursion_depth": ULTRASONIC_MAX_RECURSION_DEPTH,
    "size_selection": {
        "type": "Gaussian acceptance kernel",
        "target_nt": SIZE_SELECTION_TARGET,
        "sd_nt": SIZE_SELECTION_SD,
        "min_sequencable_nt": MIN_SEQUENCABLE_FRAGMENT_LENGTH,
    },
    "scientific_analog": "Flux Simulator nebulisation model; DOI 10.1093/nar/gks666",
    "scope_note": "approximate mechanical/ultrasonic-like shearing, not device-specific physics",
}

# PCR2: высокий предел предотвращает искусственное выравнивание численности.
LIBRARY_PCR_CYCLES = 10
LIBRARY_PCR_EFFICIENCY_MEAN = 0.85
LIBRARY_PCR_EFFICIENCY_CONCENTRATION = 80.0
LIBRARY_PCR_MAX_COPIES = 10**15

READ_BUDGET_MODE = "match_all_raw_pairs"
FIXED_READ_PAIRS = 500_000
MIXTURE_MODE = "observed_locus_depth"
RAW_PAIR_COUNTS = {
    "ERR3004229": 1_355_378,
    "ERR3004230": 1_153_931,
    "ERR3004231": 958_261,
    "ERR3004232": 1_085_144,
}

SEQUENCING_ERROR_MODEL_PROVENANCE = {
    "type": "pooled UMI-family consensus versus held-out read empirical model",
    "training_airr_dir": str(ORIGINAL_AIRR_DIR),
    "training_fastq_dir": str(PR_FASTQ_DIR),
    "conditioned_on_post_annotation_filter": False,
}

print("SOURCE STAGE:", SOURCE_STAGE)
print("TRUTH AIRR:", TRUTH_AIRR_DIR)
print("BRANCH:", BRANCH_NAME)
print("OUT_BASE:", OUT_BASE)
print("filter rules:", FILTER_RULES)


In [ ]:
# ---------------------------------------------------------------------
# Очистка сгенерированных артефактов в целевой ветви
# ---------------------------------------------------------------------
# Каталог выходных данных:
#   results/PRJEB30386/simulated/insilicoseq/
#
# При FORCE=True и CLEAN_OLD_OUTPUTS=True содержимое целевых
# выходных каталогов удаляется и создаётся заново. Исходные данные
# за пределами OUT_BASE не изменяются.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [TRUTH_DIR,PCR1_DIR,FRAGMENTATION_DIR,PCR2_DIR,ALLOCATION_DIR,FASTQ_NATIVE_DIR,FASTQ_DIR,MODEL_DIR,QC_DIR,LOGS_DIR]

    for directory in generated_dirs:
        directory = Path(directory)
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        Path(directory).mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Вспомогательные функции

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Построение эталона: ампликон PCR1 с учётом UMI

`post_annotation_filtered/airr_pass` служит эталонным входом. Условия фильтра повторно проверяются утверждениями. Дополнительно исключаются непригодные UMI и записи с N.

- **Ключ схлопывания** — (локус, наблюдаемая последовательность V…J); реальные SHM сохраняются.
- **`UMI_COLLAPSE_MODE = "umi_mode"`**: каждый UMI отдаёт одну исходную копию самому частому варианту V…J среди своих ридов. Варианты, не получившие ни одного UMI (минорные ошибочные варианты той же молекулы), исключаются из эталона.
- **`TRUTH_TEMPLATE_MODE = "amplicon_with_primers"`**: шаблон = V-leader праймер + склеенный рид (остаток leader, V…J, C-фрагмент) + revcomp(C-праймер). Праймеры были вырезаны `MaskPrimers --mode cut`; имена праймеров берутся из заголовков `pr_trimmed` (AssemblePairs их не перенёс), последовательности — из `technical_sequences.json`. UMI и Illumina-хвосты не включаются. Координаты V…J записаны в `vj_start`/`vj_end` (0-based, half-open) — именно этот интервал является эталоном для оценки сборщиков.


In [ ]:
_IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT",
          "K": "GT", "M": "AC", "B": "CGT", "D": "AGT", "H": "ACT", "V": "ACG", "N": "ACGT"}
_RC = str.maketrans("ACGT", "TGCA")

def _revcomp(s):
    return s.translate(_RC)[::-1]

def _truthy(x):
    return str(x).strip().lower() in {"t", "true", "1", "yes"}

def _falsey(x):
    return str(x).strip().lower() in {"f", "false", "0", "no"}

def _as_int(x):
    try:
        return int(x)
    except (TypeError, ValueError):
        return None

def _as_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _barcode(sequence_id):
    m = re.search(r"(?:^|\|)BARCODE=([^|\s]+)", sequence_id or "")
    return m.group(1) if m else None

def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as src:
        for chunk in iter(lambda: src.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _find_technical_sequences():
    rel = Path("docs/datasets/PRJEB30386/technical_sequences.json")
    cwd = Path.cwd().resolve()
    for root in (VOLUME, cwd, *cwd.parents, Path("/Users/epishkin/workspace/bcr-assembler")):
        if (root / rel).exists():
            return root / rel
    raise FileNotFoundError(f"{rel} not found; needed for primer sequences")

def load_gene_specific_primers():
    """Gene-specific части праймеров библиотеки (без Illumina-хвоста и C-side UMI).

    Совпадают со словарями V_LEADER_PRIMERS / CONSTANT_PRIMERS в primer_trim_human.ipynb.
    """
    path = _find_technical_sequences()
    out = {}
    for p in json.loads(path.read_text())["supplemental_table_2_library_primers"]:
        tail = p["sequence"].split("TCTTCCGATCT", 1)
        if len(tail) != 2 or not tail[1]:
            continue  # Read2U и прочие чисто технические олиго
        out[p["name"]] = tail[1][20:] if "_Ig" in p["name"] else tail[1]
    return out, path

def _pr_trimmed_primer_names(run):
    """read id -> [V primer, C primer] из заголовков pr_trimmed (AssemblePairs их не перенёс)."""
    names = {}
    for mate, slot in ((1, 0), (2, 1)):
        with gzip.open(PR_FASTQ_DIR / f"{run}_{mate}.pr.fastq.gz", "rt") as h:
            for i, line in enumerate(h):
                if i % 4:
                    continue
                rid = line[1:].split(None, 1)[0].split("|", 1)[0]
                m = re.search(r"\|PRIMER=([^|\s]+)", line)
                names.setdefault(rid, [None, None])[slot] = m.group(1) if m else None
    return names

if TRUTH_TEMPLATE_MODE == "amplicon_with_primers":
    PRIMERS, TECHNICAL_SEQUENCES_PATH = load_gene_specific_primers()
    print("gene-specific primers:", len(PRIMERS), "from", TECHNICAL_SEQUENCES_PATH)
else:
    PRIMERS, TECHNICAL_SEQUENCES_PATH = {}, None

FILTER_SUMMARY_SHA256 = _sha256(FILTER_SUMMARY_PATH)
PROVENANCE_PATH = TRUTH_DIR / "source_provenance.json"

def _validate_post_filter_row(r, run):
    """Fail loudly if airr_pass no longer matches the recorded filter contract."""
    problems = []
    expected_locus = RUN_LOCUS[run]

    v_start = _as_int(r.get("v_sequence_start"))
    j_end = _as_int(r.get("j_sequence_end"))
    vgs = _as_int(r.get("v_germline_start"))
    vi = _as_float(r.get("v_identity"))
    ji = _as_float(r.get("j_identity"))
    vs = _as_float(r.get("v_support"))
    js = _as_float(r.get("j_support"))

    if v_start is None or j_end is None or j_end - v_start + 1 < MIN_TEMPLATE_LENGTH:
        problems.append("V-J span")
    if vgs != 1:
        problems.append("V 5-prime completeness")
    if not _truthy(r.get("complete_vdj")):
        problems.append("complete_vdj")
    if str(r.get("locus", "")).strip() != expected_locus:
        problems.append("locus")
    if not _truthy(r.get("productive")):
        problems.append("productive")
    if not _falsey(r.get("stop_codon")):
        problems.append("stop_codon")
    if vi is None or vi < MIN_V_IDENTITY:
        problems.append("v_identity")
    if ji is None or ji < MIN_J_IDENTITY:
        problems.append("j_identity")
    if vs is None or vs > MAX_V_SUPPORT:
        problems.append("v_support")
    if js is None or js > MAX_J_SUPPORT:
        problems.append("j_support")

    if problems:
        raise RuntimeError(
            f"{run} airr_pass violates filter_summary contract for "
            f"{r.get('sequence_id')}: {', '.join(problems)}"
        )

def _current_provenance():
    data = {
        "dataset": DATASET,
        "source_stage": SOURCE_STAGE,
        "truth_airr_dir": str(TRUTH_AIRR_DIR),
        "filtered_fastq_dir": str(FILTERED_FASTQ_DIR),
        "filter_summary": str(FILTER_SUMMARY_PATH),
        "filter_summary_sha256": FILTER_SUMMARY_SHA256,
        "filter_rules": FILTER_RULES,
        "simulation_specific_truth_rules": {
            "require_valid_20nt_umi": REQUIRE_VALID_20NT_UMI,
            "drop_sequences_with_N": DROP_SEQUENCES_WITH_N,
            "truth_template_mode": TRUTH_TEMPLATE_MODE,
            "primer_sequences": str(TECHNICAL_SEQUENCES_PATH) if TECHNICAL_SEQUENCES_PATH else None,
            "primer_names_source": str(PR_FASTQ_DIR) if PRIMERS else None,
            "flanks": "most frequent (V primer, leader rest, C stub, C primer) among member reads; IUPAC resolved by seeded RNG; UMI not included",
            "vj_coordinates": "template_qc.tsv vj_start/vj_end, 0-based half-open",
            "collapse_key": ["locus", "VJ_sequence"],
            "umi_collapse_mode": UMI_COLLAPSE_MODE,
            "starting_copies_mode": STARTING_COPIES_MODE,
        },
        "branch": BRANCH_NAME,
        "library_input_scale": LIBRARY_INPUT_SCALE,
        "read_budget_multiplier": READ_BUDGET_MULTIPLIER,
        "seed": SEED,
        "fragmentation_model": FRAGMENTATION_MODEL_PROVENANCE,
        "sequencing_error_model": SEQUENCING_ERROR_MODEL_PROVENANCE,
    }
    return data

def _check_cached_truth_provenance():
    if not PROVENANCE_PATH.exists():
        return False
    old = json.loads(PROVENANCE_PATH.read_text())
    if old.get("filter_summary_sha256") != FILTER_SUMMARY_SHA256:
        raise RuntimeError(
            "post_annotation_filtered/filter_summary.json changed since this "
            "simulation truth was built. Use FORCE=True and CLEAN_OLD_OUTPUTS=True "
            "or choose a new branch."
        )
    if old.get("simulation_specific_truth_rules") != _current_provenance()["simulation_specific_truth_rules"]:
        raise RuntimeError("Truth construction rules changed; use FORCE=True or a new branch.")
    return True

def build_mixed_templates(force=FORCE):
    sample = MIXED_SAMPLE
    out_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    out_tsv = TEMPLATES_DIR / f"{sample}_template_qc.tsv"
    audit_path = QC_DIR / "source_truth_audit_by_run.tsv"

    if out_fa.exists() and out_tsv.exists() and audit_path.exists() and not force:
        _check_cached_truth_provenance()
        print("[skip] post-filter UMI-aware templates exist")
        return

    groups = {}
    umi_votes = defaultdict(lambda: defaultdict(int))  # run:UMI -> (locus, VJ) -> reads
    audit = []
    amplicon = TRUTH_TEMPLATE_MODE == "amplicon_with_primers"

    for run in SOURCE_RUNS:
        airr = TRUTH_AIRR_DIR / f"{run}.airr.tsv"
        filtered_fastq = FILTERED_FASTQ_DIR / f"{run}_filtered.fastq.gz"
        if not airr.exists():
            raise FileNotFoundError(f"Filtered AIRR missing: {airr}")
        if not filtered_fastq.exists():
            raise FileNotFoundError(f"Filtered FASTQ missing: {filtered_fastq}")
        primer_names = _pr_trimmed_primer_names(run) if amplicon else {}

        expected_passed = int(FILTER_SUMMARY["samples"][run]["passed"])
        a = {
            "source_run": run,
            "library": RUN_LIBRARY[run],
            "expected_locus": RUN_LOCUS[run],
            "expected_passed_from_filter_summary": expected_passed,
            "airr_pass_rows": 0,
            "simulation_eligible_rows": 0,
            "missing_barcode": 0,
            "invalid_barcode_length": 0,
            "non_acgt_barcode": 0,
            "missing_v_or_j_call": 0,
            "sequence_with_N": 0,
            "missing_primer_annotation": 0,
        }

        with open(airr, newline="") as h:
            for r in csv.DictReader(h, delimiter="\t"):
                a["airr_pass_rows"] += 1
                _validate_post_filter_row(r, run)

                full = (r.get("sequence") or "").upper()
                start = int(r["v_sequence_start"]) - 1
                end = int(r["j_sequence_end"])
                seq = full[start:end]

                # Основной фильтр гарантирует длину не менее MIN_TEMPLATE_LENGTH.
                if len(seq) < MIN_TEMPLATE_LENGTH:
                    raise RuntimeError(
                        f"{run}: cropped V-J sequence shorter than filter threshold "
                        f"for {r.get('sequence_id')}"
                    )

                if not r.get("v_call") or not r.get("j_call"):
                    a["missing_v_or_j_call"] += 1
                    continue

                if DROP_SEQUENCES_WITH_N and "N" in (full if amplicon else seq):
                    a["sequence_with_N"] += 1
                    continue

                bc = _barcode(r.get("sequence_id"))
                if REQUIRE_VALID_20NT_UMI:
                    if not bc:
                        a["missing_barcode"] += 1
                        continue
                    if len(bc) != 20:
                        a["invalid_barcode_length"] += 1
                        continue
                    if set(bc.upper()) - set("ACGT"):
                        a["non_acgt_barcode"] += 1
                        continue

                if amplicon:
                    vp, cp = primer_names.get(r["sequence_id"].split("|", 1)[0], (None, None))
                    if vp not in PRIMERS or cp not in PRIMERS:
                        a["missing_primer_annotation"] += 1
                        continue
                    flank = (vp, full[:start], full[end:], cp)
                else:
                    flank = None

                a["simulation_eligible_rows"] += 1
                locus = r["locus"]
                key = (locus, seq)
                g = groups.setdefault(
                    key,
                    {
                        "observed_reads": 0,
                        "umis": set(),
                        "runs": set(),
                        "libraries": set(),
                        "v_calls": set(),
                        "j_calls": set(),
                        "flanks": defaultdict(int),
                    },
                )
                g["observed_reads"] += 1
                g["runs"].add(run)
                g["libraries"].add(RUN_LIBRARY[run])
                g["v_calls"].add(r["v_call"])
                g["j_calls"].add(r["j_call"])
                if flank:
                    g["flanks"][flank] += 1
                if bc:
                    g["umis"].add(run + ":" + bc)
                    umi_votes[run + ":" + bc][key] += 1

        if a["airr_pass_rows"] != expected_passed:
            raise RuntimeError(
                f"{run}: airr_pass rows={a['airr_pass_rows']:,}, but "
                f"filter_summary passed={expected_passed:,}"
            )
        audit.append(a)
        del primer_names

    # Сколько исходных молекул (UMI) принадлежит каждому варианту V…J.
    if UMI_COLLAPSE_MODE == "umi_mode":
        owned = defaultdict(int)
        for votes in umi_votes.values():
            # большинство ридов UMI; при равенстве — детерминированно по последовательности
            owned[max(votes.items(), key=lambda kv: (kv[1], kv[0][1]))[0]] += 1
        for key, g in groups.items():
            g["umi_copies"] = owned.get(key, 0)
    elif UMI_COLLAPSE_MODE == "none":
        for g in groups.values():
            g["umi_copies"] = len(g["umis"])
    else:
        raise ValueError(UMI_COLLAPSE_MODE)
    del umi_votes
    variants_total = len(groups)
    groups = {k: g for k, g in groups.items() if g["umi_copies"] > 0}
    print(f"V-J variants={variants_total:,}; with own UMI molecules={len(groups):,}")

    ordered = sorted(
        groups.items(),
        key=lambda x: (
            x[0][0],
            -x[1]["umi_copies"],
            -x[1]["observed_reads"],
            x[0][1],
        ),
    )

    iupac_rng = np.random.default_rng(sample_seed(sample, 500))
    def _resolve(p):
        return "".join(c if c in "ACGT" else str(iupac_rng.choice(list(_IUPAC[c]))) for c in p)

    fields = [
        "template_id", "locus", "length", "vj_start", "vj_end", "v_primer", "c_primer",
        "observed_multiplicity", "unique_umis", "umis_seen", "source_runs", "libraries",
        "v_calls", "j_calls",
    ]
    with open(out_fa, "w") as fa, open(out_tsv, "w", newline="") as ts:
        w = csv.DictWriter(ts, fieldnames=fields, delimiter="\t")
        w.writeheader()
        for i, ((locus, seq), g) in enumerate(ordered, 1):
            tid = f"{sample}_{locus}_tpl_{i:07d}"
            if amplicon:
                vp, lead, cstub, cp = max(g["flanks"].items(), key=lambda kv: kv[1])[0]
                left = _resolve(PRIMERS[vp]) + lead
                template = left + seq + cstub + _revcomp(_resolve(PRIMERS[cp]))
            else:
                vp = cp = ""
                left, template = "", seq
            fa.write(f">{tid}\n{template}\n")
            w.writerow({
                "template_id": tid,
                "locus": locus,
                "length": len(template),
                "vj_start": len(left),
                "vj_end": len(left) + len(seq),
                "v_primer": vp,
                "c_primer": cp,
                "observed_multiplicity": g["observed_reads"],
                "unique_umis": g["umi_copies"],
                "umis_seen": len(g["umis"]),
                "source_runs": ";".join(sorted(g["runs"])),
                "libraries": ";".join(sorted(g["libraries"])),
                "v_calls": ";".join(sorted(g["v_calls"])),
                "j_calls": ";".join(sorted(g["j_calls"])),
            })

    with open(audit_path, "w", newline="") as h:
        fields = list(audit[0])
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows(audit)

    PROVENANCE_PATH.write_text(json.dumps(_current_provenance(), indent=2) + "\n")
    print(
        f"post-filter mixed templates={len(ordered):,}; "
        f"loci={sorted({k[0] for k in groups})}"
    )

build_mixed_templates()


## 5. Сводка контроля качества дедупликации

In [ ]:
q=TEMPLATES_DIR/f"{MIXED_SAMPLE}_template_qc.tsv"; rows=list(csv.DictReader(open(q),delimiter="\t")); loci=sorted({r["locus"] for r in rows}); summary=[]
for locus in loci:
    x=[r for r in rows if r["locus"]==locus]; summary.append({"locus":locus,"unique_templates":len(x),"observed_reads":sum(int(r["observed_multiplicity"]) for r in x),"unique_umis":sum(int(r["unique_umis"]) for r in x)})
out=QC_DIR/"template_summary_by_locus.tsv"
with open(out,"w",newline="") as h:
    w=csv.DictWriter(h,fieldnames=list(summary[0]),delimiter="\t"); w.writeheader(); w.writerows(summary)
print(*summary,sep="\n"); print("wrote",out)


## 6. Объединённая эмпирическая модель на основе UMI-консенсуса PE150

Биологический эталон берётся из `post_annotation_filtered/airr_pass`, а модель ошибок секвенирования обучается по исходной AIRR-аннотации и соответствующим парам `pr_trimmed`. Такое разделение исключает влияние последующей фильтрации аннотаций на профиль ошибок.


In [ ]:
import sqlite3

MIN_UMI_FAMILY = 3
CONSENSUS_MIN_FREQ = 0.8
MAX_V_GERMLINE_START = 15
MIN_MAPQ = 10

FAMILY_DB = MODEL_DIR / "umi_family_members.sqlite"
CONSENSUS_CANDIDATES = MODEL_DIR / "umi_consensus_candidates.fasta"
FAMILY_MANIFEST = MODEL_DIR / "umi_family_manifest.tsv"
CONSENSUS_AIRR = MODEL_DIR / "umi_consensus_candidates.airr.tsv"
VERIFIED_CONSENSUS = MODEL_DIR / "umi_consensus_verified.fasta"
VERIFIED_MANIFEST = MODEL_DIR / "umi_consensus_verified.tsv"
HELDOUT_R1 = MODEL_DIR / "heldout_R1.fastq.gz"
HELDOUT_R2 = MODEL_DIR / "heldout_R2.fastq.gz"
REF_INDEX = MODEL_DIR / "umi_consensus_bt2"
RAW_BAM = MODEL_DIR / "heldout_vs_consensus.raw.bam"
FILTERED_BAM = MODEL_DIR / "heldout_vs_consensus.filtered.bam"
CUSTOM_MODEL_PREFIX = MODEL_DIR / f"{DATASET}_pooled_umi_consensus_{TARGET_READ_LENGTH}bp"
CUSTOM_MODEL_NPZ = Path(str(CUSTOM_MODEL_PREFIX) + ".npz")

def gene1(x):
    return (x or "").split(",")[0].split("*")[0]

def read_id(x):
    return (x or "").split()[0].split("|")[0].removesuffix("/1").removesuffix("/2")

def family_id(run, umi, v, j, cdr3):
    return "FAM_" + hashlib.sha1(
        "\0".join((run, umi, v, j, cdr3)).encode()
    ).hexdigest()[:20]

def majority_consensus(seqs, min_freq=CONSENSUS_MIN_FREQ):
    if not seqs or len({len(s) for s in seqs}) != 1:
        return None
    out = []
    for chars in zip(*seqs):
        counts = {b: chars.count(b) for b in "ACGT"}
        b, n = max(counts.items(), key=lambda x: x[1])
        out.append(b if n / len(chars) >= min_freq else "N")
    ans = "".join(out)
    return ans if "N" not in ans else None

def prepare_family_consensus(force=FORCE):
    if CONSENSUS_CANDIDATES.exists() and FAMILY_MANIFEST.exists() and not force:
        print("[skip] family consensus candidates")
        return

    FAMILY_DB.unlink(missing_ok=True)
    con = sqlite3.connect(FAMILY_DB)
    con.execute(
        "create table members("
        "family text,run text,umi text,read_id text,locus text,"
        "v_gene text,j_gene text,cdr3 text,seq text)"
    )
    batch = []

    # Здесь используется ORIGINAL_AIRR_DIR: стадия оценивает
    # ошибки платформы секвенирования, а не отфильтрованный биологический эталон.
    for run in SOURCE_RUNS:
        with open(ORIGINAL_AIRR_DIR / f"{run}.airr.tsv", newline="") as h:
            for r in csv.DictReader(h, delimiter="\t"):
                umi = _barcode(r.get("sequence_id"))
                locus = r.get("locus") or ""
                try:
                    a = int(r.get("v_sequence_start") or 0) - 1
                    b = int(r.get("j_sequence_end") or 0)
                    vgs = int(r.get("v_germline_start") or 999)
                except ValueError:
                    continue

                if not (
                    umi
                    and len(umi) == 20
                    and set(umi.upper()) <= set("ACGT")
                    and locus == RUN_LOCUS[run]
                    and _truthy(r.get("productive"))
                    and _truthy(r.get("complete_vdj"))
                    and not _truthy(r.get("stop_codon"))
                    and vgs <= MAX_V_GERMLINE_START
                    and b > a
                ):
                    continue

                seq = (r.get("sequence") or "")[a:b].upper()
                v = gene1(r.get("v_call"))
                j = gene1(r.get("j_call"))
                cdr3 = r.get("cdr3") or ""
                if len(seq) < 180 or "N" in seq or not (v and j and cdr3):
                    continue

                fid = family_id(run, umi, v, j, cdr3)
                batch.append(
                    (fid, run, umi, read_id(r.get("sequence_id")),
                     locus, v, j, cdr3, seq)
                )
                if len(batch) >= 10000:
                    con.executemany(
                        "insert into members values(?,?,?,?,?,?,?,?,?)", batch
                    )
                    con.commit()
                    batch.clear()

    if batch:
        con.executemany("insert into members values(?,?,?,?,?,?,?,?,?)", batch)
        con.commit()

    con.execute("create index family_idx on members(family)")
    con.commit()

    fields = [
        "family_id", "run", "umi", "locus", "v_gene", "j_gene", "cdr3",
        "family_size", "consensus_members", "heldout_read_id",
        "consensus_length",
    ]
    with open(Path(str(CONSENSUS_CANDIDATES) + ".tmp"), "w") as fa, \
         open(Path(str(FAMILY_MANIFEST) + ".tmp"), "w", newline="") as mh:
        w = csv.DictWriter(mh, fieldnames=fields, delimiter="\t")
        w.writeheader()
        accepted = 0

        for fid, n in con.execute(
            "select family,count(*) from members group by family "
            "having count(*)>=? order by family",
            (MIN_UMI_FAMILY,),
        ):
            rows = con.execute(
                "select run,umi,read_id,locus,v_gene,j_gene,cdr3,seq "
                "from members where family=? order by read_id",
                (fid,),
            ).fetchall()
            held = rows[-1]
            cons = majority_consensus([x[-1] for x in rows[:-1]])
            if cons is None:
                continue
            fa.write(f">{fid}\n{cons}\n")
            w.writerow(dict(zip(
                fields,
                [
                    fid, held[0], held[1], held[3], held[4], held[5], held[6],
                    n, n - 1, held[2], len(cons),
                ],
            )))
            accepted += 1

    con.close()
    Path(str(CONSENSUS_CANDIDATES) + ".tmp").replace(CONSENSUS_CANDIDATES)
    Path(str(FAMILY_MANIFEST) + ".tmp").replace(FAMILY_MANIFEST)
    print("consensus candidates", accepted)

prepare_family_consensus()


In [ ]:
# Независимая germline-проверка оценивает согласованность аннотации, но не задаёт ошибки секвенирования.
IGDATA=Path(_CONDA_ENV)/"share/igblast"; DB=IGDATA/"database/airr_c_human"; V_DB=DB/"airr_c_human_ig.V"; D_DB=DB/"airr_c_human_igh.D"; J_DB=DB/"airr_c_human_ig.J"; AUX=IGDATA/"optional_file/human_gl.aux"
def annotate_consensus(force=FORCE):
    if CONSENSUS_AIRR.exists() and not force: print("[skip] consensus IgBLAST"); return
    tmp=Path(str(CONSENSUS_AIRR)+".tmp"); cmd=["igblastn","-germline_db_V",str(V_DB),"-germline_db_D",str(D_DB),"-germline_db_J",str(J_DB),"-organism","human","-ig_seqtype","Ig","-domain_system","imgt","-auxiliary_data",str(AUX),"-query",str(CONSENSUS_CANDIDATES),"-outfmt","19","-num_threads",str(NPROC),"-out",str(tmp)]
    run_with_heartbeat(cmd,LOGS_DIR/"umi_consensus_igblast.log"); tmp.replace(CONSENSUS_AIRR)
def verify_consensus(force=FORCE):
    if VERIFIED_CONSENSUS.exists() and VERIFIED_MANIFEST.exists() and not force: print("[skip] verified consensus"); return
    meta={r["family_id"]:r for r in csv.DictReader(open(FAMILY_MANIFEST),delimiter="\t")}; seqs=dict(iter_fasta(CONSENSUS_CANDIDATES)); kept=[]
    for r in csv.DictReader(open(CONSENSUS_AIRR),delimiter="\t"):
        m=meta.get(r["sequence_id"])
        if not m: continue
        if not (_truthy(r.get("productive")) and _truthy(r.get("complete_vdj")) and not _truthy(r.get("stop_codon")) and r.get("locus")==m["locus"] and gene1(r.get("v_call"))==m["v_gene"] and gene1(r.get("j_call"))==m["j_gene"]): continue
        kept.append({**m,"consensus_v_identity":r.get("v_identity",""),"consensus_j_identity":r.get("j_identity","")})
    by_run=defaultdict(list)
    for r in kept: by_run[r["run"]].append(r)
    if set(by_run)!=set(SOURCE_RUNS): raise RuntimeError(f"missing verified source run: {sorted(by_run)}")
    balanced_n=min(len(by_run[run]) for run in SOURCE_RUNS)
    kept=[r for run in SOURCE_RUNS for r in sorted(by_run[run],key=lambda x:x["family_id"])[:balanced_n]]
    fields=list(kept[0]); tmpfa=Path(str(VERIFIED_CONSENSUS)+".tmp"); tmpts=Path(str(VERIFIED_MANIFEST)+".tmp")
    with open(tmpfa,"w") as fa,open(tmpts,"w",newline="") as th:
        w=csv.DictWriter(th,fieldnames=fields,delimiter="\t"); w.writeheader()
        for r in kept: fa.write(f">{r['family_id']}\n{seqs[r['family_id']]}\n"); w.writerow(r)
    tmpfa.replace(VERIFIED_CONSENSUS); tmpts.replace(VERIFIED_MANIFEST); print("balanced germline-verified families",len(kept),"per run",balanced_n)
annotate_consensus(); verify_consensus()


In [ ]:
def fastq_records(path):
    with open_text(path,"rt") as h:
        while True:
            head=h.readline()
            if not head: return
            seq=h.readline(); plus=h.readline(); qual=h.readline()
            if not qual: raise ValueError(f"truncated {path}")
            yield head,seq,plus,qual
def extract_heldout(force=FORCE):
    if HELDOUT_R1.exists() and HELDOUT_R2.exists() and not force: print("[skip] heldout FASTQ"); return
    rows=list(csv.DictReader(open(VERIFIED_MANIFEST),delimiter="\t")); wanted=defaultdict(set)
    for r in rows: wanted[r["run"]].add(r["heldout_read_id"])
    found=set(); t1=Path(str(HELDOUT_R1)+".tmp"); t2=Path(str(HELDOUT_R2)+".tmp")
    with gzip.open(t1,"wt") as o1,gzip.open(t2,"wt") as o2:
        for run in SOURCE_RUNS:
            p1=PR_FASTQ_DIR/f"{run}_1.pr.fastq.gz"; p2=PR_FASTQ_DIR/f"{run}_2.pr.fastq.gz"
            for a,b in zip(fastq_records(p1),fastq_records(p2)):
                i1=read_id(a[0][1:]); i2=read_id(b[0][1:])
                if i1!=i2: raise ValueError(f"mate ID mismatch {i1} {i2}")
                if i1 in wanted[run]: o1.writelines(a); o2.writelines(b); found.add((run,i1))
    expected={(r["run"],r["heldout_read_id"]) for r in rows}; missing=expected-found
    if missing: raise RuntimeError(f"heldout reads missing: {len(missing)}")
    t1.replace(HELDOUT_R1); t2.replace(HELDOUT_R2); print("heldout pairs",len(found))

def build_and_align(force=FORCE):
    marker=Path(str(REF_INDEX)+".1.bt2"); marker_l=Path(str(REF_INDEX)+".1.bt2l")
    if not (marker.exists() or marker_l.exists()) or force: run_with_heartbeat(["bowtie2-build","--threads",str(NPROC),str(VERIFIED_CONSENSUS),str(REF_INDEX)],LOGS_DIR/"umi_consensus_bowtie2_build.log")
    sam=MODEL_DIR/"heldout_vs_consensus.sam"
    if not RAW_BAM.exists() or force:
        run_with_heartbeat(["bowtie2","--local","--no-mixed","--no-discordant","--trim-to",str(TARGET_READ_LENGTH),"-p",str(NPROC),"-x",str(REF_INDEX),"-1",str(HELDOUT_R1),"-2",str(HELDOUT_R2),"-S",str(sam)],LOGS_DIR/"heldout_bowtie2.log")
        run_with_heartbeat(["samtools","sort","-@",str(NPROC),"-o",str(RAW_BAM),str(sam)],LOGS_DIR/"heldout_samtools_sort.log"); subprocess.run(["samtools","index",str(RAW_BAM)],check=True); sam.unlink(missing_ok=True)
extract_heldout(); build_and_align()


import pysam
def filter_expected_alignments(force=FORCE):
    if FILTERED_BAM.exists() and Path(str(FILTERED_BAM)+".bai").exists() and not force: print("[skip] filtered BAM"); return
    expected={r["heldout_read_id"]:r["family_id"] for r in csv.DictReader(open(VERIFIED_MANIFEST),delimiter="\t")}; tmp=Path(str(FILTERED_BAM)+".tmp")
    kept=seen=0
    with pysam.AlignmentFile(RAW_BAM,"rb") as src,pysam.AlignmentFile(tmp,"wb",template=src) as dst:
        for rec in src:
            seen+=1; target=expected.get(read_id(rec.query_name))
            if target and not rec.is_unmapped and rec.mapping_quality>=MIN_MAPQ and rec.reference_name==target: dst.write(rec); kept+=1
    pysam.sort("-o",str(FILTERED_BAM),str(tmp)); tmp.unlink(missing_ok=True); pysam.index(str(FILTERED_BAM)); print({"alignments_seen":seen,"expected_alignments_kept":kept})
def fit_model(force=FORCE):
    if CUSTOM_MODEL_NPZ.exists() and not force: print("[skip]",CUSTOM_MODEL_NPZ); return
    run_with_heartbeat(["iss","model","--debug","-b",str(FILTERED_BAM),"-o",str(CUSTOM_MODEL_PREFIX)],LOGS_DIR/"iss_model.log")
    if not CUSTOM_MODEL_NPZ.exists(): raise RuntimeError("ISS model missing")
    from iss.error_models.kde import KDErrorModel
    em=KDErrorModel(str(CUSTOM_MODEL_NPZ)); assert em.read_length==TARGET_READ_LENGTH; print("custom pooled model",CUSTOM_MODEL_NPZ,em.read_length)
filter_expected_alignments(); fit_model()


## 7. Бюджет смешанного секвенирования

Один синтетический FASTQ содержит все локусы. Глубина по умолчанию равна сумме глубин четырёх исходных индексированных библиотек. Бюджеты локусов воспроизводят их вычислительное объединение; внутри локуса численность определяется UMI и моделью PCR. Нативное спаривание тяжёлой и лёгкой цепей не моделируется.


In [ ]:
def get_read_budget(sample):
    if READ_BUDGET_MODE=="fixed": return int(FIXED_READ_PAIRS)
    if READ_BUDGET_MODE=="match_all_raw_pairs":
        total=0
        for run in SOURCE_RUNS:
            n1=count_fastq(RAW_FASTQ_DIR/f"{run}_1.fastq.gz"); n2=count_fastq(RAW_FASTQ_DIR/f"{run}_2.fastq.gz")
            if n1!=n2: raise ValueError(f"{run}: mate count mismatch")
            total+=n1
        return int(round(total*READ_BUDGET_MULTIPLIER))
    raise ValueError(READ_BUDGET_MODE)
READ_BUDGETS={MIXED_SAMPLE:get_read_budget(MIXED_SAMPLE)}; print(READ_BUDGETS)


## 8. PCR1 для наблюдаемых шаблонов V…J


In [ ]:
def load_dedup_templates(sample):
    fa=TEMPLATES_DIR/f"{sample}_templates.fasta"; qc=TEMPLATES_DIR/f"{sample}_template_qc.tsv"; meta={r["template_id"]:r for r in csv.DictReader(open(qc),delimiter="\t")}
    return [{"template_id":tid,"sequence":seq,"length":len(seq),"vj_start":int(meta[tid].get("vj_start") or 0),"vj_end":int(meta[tid].get("vj_end") or len(seq)),"observed_multiplicity":int(meta[tid]["observed_multiplicity"]),"unique_umis":int(meta[tid]["unique_umis"]),"locus":meta[tid]["locus"]} for tid,seq in iter_fasta(fa)]
def starting_copies(row):
    if STARTING_COPIES_MODE=="unique_umis": return max(1,row["unique_umis"])
    if STARTING_COPIES_MODE=="one_per_unique": return 1
    if STARTING_COPIES_MODE=="observed_multiplicity": return row["observed_multiplicity"]
    raise ValueError(STARTING_COPIES_MODE)
def branching_pcr(n0,efficiency,cycles,rng):
    n=int(n0)
    for _ in range(cycles):
        if n<=0:return 0
        n+=int(rng.binomial(n,efficiency))
        if n>=PCR_MAX_COPIES:return int(PCR_MAX_COPIES)
    return n
def simulate_pcr_pool(sample):
    rng=np.random.default_rng(sample_seed(sample,1000)); rows=load_dedup_templates(sample); mean=PCR_EFFICIENCY_MEAN; conc=PCR_EFFICIENCY_CONCENTRATION; alpha,beta=(mean*conc,(1-mean)*conc) if mean<1 else (None,None)
    for r in rows:
        p=1. if mean==1 else float(rng.beta(alpha,beta)); n0=starting_copies(r); r.update(starting_copies=n0,pcr_efficiency=p,pcr_copies=branching_pcr(n0,p,PCR_CYCLES,rng))
    return rows
_test_rng=np.random.default_rng(1); assert branching_pcr(1,1.,10,_test_rng)==2**10; assert branching_pcr(1,0.,10,_test_rng)==1; print("branching PCR smoke tests: OK")


## 9. Фрагментация: ultrasonic-like mechanical shearing

Это отдельная приближённая модель **ультразвуковой/механической фрагментации**, а не симуляция кавитации или конкретного прибора Covaris.

После выборки конечного aliquot из PCR1-пула каждая молекула проходит рекурсивный процесс:

- вероятность ещё одного разрыва зависит от текущей длины фрагмента: `p_break = exp(-lambda / L)`;
- потенциальная точка разрыва выбирается из дискретного Gaussian-распределения вокруг середины текущего фрагмента;
- дочерние фрагменты снова могут разрушаться, максимум до `ULTRASONIC_MAX_RECURSION_DEPTH`;
- после завершения shearing применяется тот же отдельный мягкий size-selection вокруг `SIZE_SELECTION_TARGET` nt (по умолчанию 250) (`SD=40`).

Такая абстракция основана на подходе Flux Simulator к mechanical/nebulisation fragmentation: size-dependent breaking probability + breakpoint distribution around molecular midpoint, а затем отдельный size selection (Griebel et al., 2012, DOI `10.1093/nar/gks666`). Параметры здесь масштабированы под короткие V–J ампликоны и должны рассматриваться как калибруемые, а не как физические константы ультразвука.


In [ ]:
def _size_selection_probability(length):
    """Probability that a physical fragment enters the PCR2 library (scalar or array)."""
    length = np.asarray(length, dtype=float)
    z = (length - SIZE_SELECTION_TARGET) / SIZE_SELECTION_SD
    return np.where(length < MIN_SEQUENCABLE_FRAGMENT_LENGTH, 0.0, np.exp(-0.5 * z * z))


def _allocate_library_input(rows_pcr1, rng):
    starting_total = sum(max(0, int(r["starting_copies"])) for r in rows_pcr1)
    pcr1_total = sum(max(0, int(r["pcr_copies"])) for r in rows_pcr1)
    if starting_total <= 0 or pcr1_total <= 0:
        raise RuntimeError("empty PCR1 pool")

    requested = max(1, int(round(starting_total * LIBRARY_INPUT_SCALE)))
    n_input = min(requested, pcr1_total)
    weights = np.asarray([max(0, int(r["pcr_copies"])) for r in rows_pcr1], dtype=float)
    probs = weights / weights.sum()
    allocations = rng.multinomial(n_input, probs)
    return allocations, int(n_input), int(pcr1_total), int(starting_total)


def _ultrasonic_break_probability(length):
    """Flux-inspired size-dependent probability of another mechanical break."""
    length = int(length)
    if length <= 1:
        return 0.0
    return float(math.exp(-ULTRASONIC_LIMITING_LENGTH / length))


def _midpoint_break_weights(length):
    """Discrete Gaussian weights for inter-base breakpoints around midpoint."""
    positions = np.arange(1, int(length), dtype=float)
    if not len(positions):
        return positions.astype(int), np.asarray([], dtype=float)
    sigma = max(1.0, ULTRASONIC_BREAKPOINT_SD_FRACTION * float(length))
    z = (positions - float(length) / 2.0) / sigma
    weights = np.exp(-0.5 * z * z)
    weights /= weights.sum()
    return positions.astype(int), weights


def _recursive_shear_template(template_id, locus, seq, n_input, rng):
    """Aggregate recursive mechanical-shearing events for one template."""
    tlen = len(seq)
    active = {(0, tlen): int(n_input)}
    terminal = defaultdict(int)
    fragmentation_events = 0

    for _depth in range(ULTRASONIC_MAX_RECURSION_DEPTH):
        if not active:
            break
        next_active = defaultdict(int)
        for (start, end), n in active.items():
            flen = end - start
            p_break = _ultrasonic_break_probability(flen)
            n_break = int(rng.binomial(int(n), p_break)) if p_break > 0 else 0
            n_intact = int(n) - n_break
            if n_intact:
                terminal[(start, end)] += n_intact
            if not n_break:
                continue

            fragmentation_events += n_break
            positions, weights = _midpoint_break_weights(flen)
            if not len(positions):
                terminal[(start, end)] += n_break
                continue

            split_counts = rng.multinomial(n_break, weights)
            for rel, count in zip(positions, split_counts):
                count = int(count)
                if count <= 0:
                    continue
                cut = start + int(rel)
                next_active[(start, cut)] += count
                next_active[(cut, end)] += count
        active = next_active

    # Molecules still active at the recursion cap are terminal, not discarded.
    for interval, n in active.items():
        terminal[interval] += int(n)

    return terminal, int(fragmentation_events)


def enumerate_fragments(sample, rows_pcr1, rng):
    """Approximate ultrasonic/mechanical shearing without simulating cavitation.

    The abstraction follows the style used for nebulisation in Flux Simulator:
    longer fragments are more likely to break again; breakpoints are sampled
    around the fragment midpoint; the resulting pool is subsequently size-selected.
    """
    input_alloc, library_input_molecules, pcr1_total, starting_total = _allocate_library_input(rows_pcr1, rng)

    fragments = []
    input_nt_mass = terminal_nt_mass = 0
    terminal_fragment_molecules = terminal_fragment_species = 0
    retained_fragment_molecules = discarded_fragment_molecules = 0
    fragmentation_events = 0
    library_input_templates = 0

    for r, n_input in zip(rows_pcr1, input_alloc):
        n_input = int(n_input)
        if n_input <= 0:
            continue
        library_input_templates += 1
        seq = r["sequence"]
        tlen = int(r["length"])
        input_nt_mass += n_input * tlen

        terminal, n_breaks = _recursive_shear_template(
            r["template_id"], r["locus"], seq, n_input, rng
        )
        fragmentation_events += n_breaks
        # Size selection per template: unselected pieces are never materialised.
        for (start, end), n in terminal.items():
            n = int(n)
            flen = end - start
            terminal_fragment_molecules += n
            terminal_fragment_species += 1
            terminal_nt_mass += flen * n
            p_keep = float(_size_selection_probability(flen))
            kept = int(rng.binomial(n, p_keep)) if p_keep > 0 else 0
            discarded_fragment_molecules += n - kept
            if kept <= 0:
                continue
            retained_fragment_molecules += kept
            fragments.append({
                "fragment_id": f"{r['template_id']}_us_{start}_{end}",
                "template_id": r["template_id"],
                "locus": r["locus"],
                "template_length": int(tlen),
                "fragment_start": int(start),
                "fragment_length": int(flen),
                "sequence": seq[start:end],
                "fragment_input_copies": int(kept),
            })

    if terminal_nt_mass != input_nt_mass:
        raise AssertionError(
            f"ultrasonic-like nucleotide-mass conservation failed: input={input_nt_mass}, terminal={terminal_nt_mass}"
        )

    if not fragments:
        raise RuntimeError(f"{sample}: ultrasonic-like size selection removed the entire library")

    summary = {
        "sample": sample,
        "fragmentation_model": FRAGMENTATION_MODEL,
        "pcr1_molecules_total": int(pcr1_total),
        "starting_copies_total": int(starting_total),
        "library_input_molecules": int(library_input_molecules),
        "library_input_templates": int(library_input_templates),
        "fragmentation_events": int(fragmentation_events),
        "terminal_fragment_molecules": int(terminal_fragment_molecules),
        "terminal_fragment_species": int(terminal_fragment_species),
        "retained_fragment_molecules": int(retained_fragment_molecules),
        "retained_fragment_species": int(len(fragments)),
        "discarded_fragment_molecules": int(discarded_fragment_molecules),
        "input_nt_mass": int(input_nt_mass),
        "terminal_nt_mass": int(terminal_nt_mass),
        "fragmentation_nt_conserved": bool(input_nt_mass == terminal_nt_mass),
        "size_selection_target_nt": int(SIZE_SELECTION_TARGET),
        "size_selection_sd_nt": int(SIZE_SELECTION_SD),
        "min_sequencable_fragment_nt": int(MIN_SEQUENCABLE_FRAGMENT_LENGTH),
        "ultrasonic_limiting_length_nt": float(ULTRASONIC_LIMITING_LENGTH),
        "ultrasonic_breakpoint_sd_fraction": float(ULTRASONIC_BREAKPOINT_SD_FRACTION),
        "ultrasonic_max_recursion_depth": int(ULTRASONIC_MAX_RECURSION_DEPTH),
    }
    return fragments, summary


## 10. PCR2 для фрагментов

PCR2 применяется к сохранённым фрагментам и формирует отдельную таблицу стадии.


In [ ]:
def branching_pcr_vectorized(n0, efficiency, cycles, max_copies, rng):
    n = np.asarray(n0, dtype=np.int64).copy()
    eff = np.asarray(efficiency, dtype=np.float64)
    for _ in range(cycles):
        active = n > 0
        if not active.any():
            break
        new = np.zeros_like(n)
        new[active] = rng.binomial(n[active], eff[active])
        n = n + new
        np.minimum(n, max_copies, out=n)
    return n


def run_pcr2_on_fragments(fragments, rng):
    mean = LIBRARY_PCR_EFFICIENCY_MEAN
    conc = LIBRARY_PCR_EFFICIENCY_CONCENTRATION
    if mean == 1:
        efficiency = np.ones(len(fragments))
    else:
        alpha, beta = mean * conc, (1 - mean) * conc
        efficiency = rng.beta(alpha, beta, size=len(fragments))

    n0 = np.array([f["fragment_input_copies"] for f in fragments], dtype=np.int64)
    copies = branching_pcr_vectorized(
        n0, efficiency, LIBRARY_PCR_CYCLES, LIBRARY_PCR_MAX_COPIES, rng
    )
    for f, eff, cp in zip(fragments, efficiency, copies):
        f["library_pcr_efficiency"] = float(eff)
        f["library_pcr2_copies"] = int(cp)
    return fragments

_test_rng = np.random.default_rng(1)
_vec = branching_pcr_vectorized(
    np.array([1]), np.array([1.0]), 10, 10**15, _test_rng
)
assert int(_vec[0]) == 2**10
print("vectorized branching PCR smoke test: OK")


## 11. Сохранение PCR1 → фрагментация → PCR2 → точное распределение ридов


In [ ]:
def write_dict_rows(path, rows, fields):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields, delimiter="\t")
        w.writeheader()
        w.writerows({k: r[k] for k in fields} for r in rows)
    tmp.replace(path)

def read_dict_rows(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

def run_pcr1_stage(sample, force=FORCE):
    out = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR1", out)
        return
    rows = simulate_pcr_pool(sample)
    fields = [
        "template_id", "locus", "length", "observed_multiplicity",
        "unique_umis", "starting_copies", "pcr_efficiency", "pcr_copies",
    ]
    write_dict_rows(out, rows, fields)
    print("PCR1 rows:", len(rows), "->", out)

def run_fragmentation_stage(sample, force=FORCE):
    inp = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    out = FRAGMENTATION_DIR / f"{sample}_fragments.tsv"
    summary_out = FRAGMENTATION_DIR / f"{sample}_fragmentation_summary.tsv"

    if out.exists() and summary_out.exists() and not force:
        print("[skip] fragmentation", out)
        return

    rows = []
    for row in read_dict_rows(inp):
        rows.append({
            **row,
            "length": int(row["length"]),
            "starting_copies": int(row["starting_copies"]),
            "pcr_copies": int(row["pcr_copies"]),
            "sequence": None,
        })

    sequences = dict(iter_fasta(TRUTH_DIR / f"{sample}_templates.fasta"))
    for row in rows:
        row["sequence"] = sequences[row["template_id"]]

    fragments, frag_summary = enumerate_fragments(
        sample,
        rows,
        np.random.default_rng(sample_seed(sample, 2000)),
    )

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "sequence", "fragment_input_copies",
    ]
    write_dict_rows(out, fragments, fields)
    write_dict_rows(summary_out, [frag_summary], list(frag_summary))

    print("fragmentation model:", FRAGMENTATION_MODEL)
    print("fragmentation rows:", len(fragments), "->", out)
    print("fragmentation summary:", frag_summary)


def run_pcr2_stage(sample, force=FORCE):
    inp = FRAGMENTATION_DIR / f"{sample}_fragments.tsv"
    out = PCR2_DIR / f"{sample}_pcr2_pool.tsv"
    if out.exists() and not force:
        print("[skip] PCR2", out)
        return

    frags = read_dict_rows(inp)
    for f in frags:
        for k in (
            "template_length", "fragment_start", "fragment_length",
            "fragment_input_copies",
        ):
            f[k] = int(f[k])

    frags = run_pcr2_on_fragments(
        frags,
        np.random.default_rng(sample_seed(sample, 3000)),
    )
    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "sequence", "fragment_input_copies",
        "library_pcr_efficiency", "library_pcr2_copies",
    ]
    write_dict_rows(out, frags, fields)
    print("PCR2 rows:", len(frags), "->", out)

def run_allocation_stage(sample, force=FORCE):
    inp = PCR2_DIR / f"{sample}_pcr2_pool.tsv"
    out_counts = ALLOCATION_DIR / f"{sample}_read_counts.tsv"
    out_fa = ALLOCATION_DIR / f"{sample}_selected_fragments.fasta"
    out_qc = ALLOCATION_DIR / f"{sample}_allocation.tsv"
    if out_counts.exists() and out_fa.exists() and out_qc.exists() and not force:
        print("[skip] allocation")
        return

    frags = read_dict_rows(inp)
    for f in frags:
        for k in (
            "template_length", "fragment_start", "fragment_length",
            "fragment_input_copies", "library_pcr2_copies",
        ):
            f[k] = int(f[k])
        f["library_pcr_efficiency"] = float(f["library_pcr_efficiency"])

    weights = np.asarray([f["library_pcr2_copies"] for f in frags], dtype=float)
    if not len(weights) or weights.sum() <= 0:
        raise RuntimeError("PCR2 pool is empty")

    target = int(READ_BUDGETS[sample])
    allocated = np.zeros(len(frags), dtype=np.int64)
    rng = np.random.default_rng(sample_seed(sample, 4000))

    if MIXTURE_MODE == "umi_pool":
        allocated = rng.multinomial(target, weights / weights.sum())
    elif MIXTURE_MODE == "observed_locus_depth":
        depth = {
            l: sum(
                RAW_PAIR_COUNTS[r]
                for r in SOURCE_RUNS
                if RUN_LOCUS[r] == l
            )
            for l in ("IGH", "IGK", "IGL")
        }
        exact = {l: target * n / sum(depth.values()) for l, n in depth.items()}
        budget = {l: int(np.floor(v)) for l, v in exact.items()}
        for l in sorted(
            exact,
            key=lambda x: exact[x] - budget[x],
            reverse=True,
        )[: target - sum(budget.values())]:
            budget[l] += 1

        for l, n in budget.items():
            idx = np.asarray([i for i, f in enumerate(frags) if f["locus"] == l])
            if not len(idx):
                raise RuntimeError(f"No simulated fragments for locus {l}")
            lw = weights[idx]
            allocated[idx] = rng.multinomial(n, lw / lw.sum())
        print("locus budgets", budget)
    else:
        raise ValueError(MIXTURE_MODE)

    with open(Path(str(out_counts) + ".tmp"), "w", newline="") as ch, \
         open(Path(str(out_fa) + ".tmp"), "w") as fh:
        w = csv.writer(ch, delimiter="\t")
        for f, n in zip(frags, allocated):
            f["simulated_read_pairs"] = int(n)
            if n:
                # InSilicoSeq readcount_file ожидает число ридов, поэтому число пар умножается на 2.
                w.writerow([f["fragment_id"], int(n) * 2])
                fh.write(f">{f['fragment_id']}\n{f['sequence']}\n")

    Path(str(out_counts) + ".tmp").replace(out_counts)
    Path(str(out_fa) + ".tmp").replace(out_fa)

    fields = [
        "fragment_id", "template_id", "locus", "template_length",
        "fragment_start", "fragment_length", "fragment_input_copies",
        "library_pcr_efficiency", "library_pcr2_copies",
        "simulated_read_pairs",
    ]
    write_dict_rows(out_qc, frags, fields)
    assert int(allocated.sum()) == target
    print("allocated", target, "pairs")

for sample in SAMPLES:
    run_pcr1_stage(sample)
    run_fragmentation_stage(sample)
    run_pcr2_stage(sample)
    run_allocation_stage(sample)


## 12. Контроль качества PCR1, фрагментации и PCR2


In [ ]:
pcr_summary = []
for sample in SAMPLES:
    pcr1_rows = read_dict_rows(PCR1_DIR / f"{sample}_pcr1_pool.tsv")
    frag_rows = read_dict_rows(FRAGMENTATION_DIR / f"{sample}_fragments.tsv")
    frag_summary = read_dict_rows(FRAGMENTATION_DIR / f"{sample}_fragmentation_summary.tsv")[0]
    pcr2_rows = read_dict_rows(PCR2_DIR / f"{sample}_pcr2_pool.tsv")

    pcr1_n = len(pcr1_rows)
    pcr1_molecules = sum(int(r["pcr_copies"]) for r in pcr1_rows)
    fragment_input_molecules = sum(int(r["fragment_input_copies"]) for r in frag_rows)

    if str(frag_summary["fragmentation_nt_conserved"]).lower() not in {"true", "1"}:
        raise AssertionError("fragmentation nucleotide-mass conservation QC failed")
    if int(frag_summary["retained_fragment_molecules"]) != fragment_input_molecules:
        raise AssertionError("fragmentation retained-molecule accounting QC failed")

    copies2 = [int(r["library_pcr2_copies"]) for r in pcr2_rows]
    eff2 = [float(r["library_pcr_efficiency"]) for r in pcr2_rows]

    allocated_n = allocated_nonzero = allocated_total = 0
    with open(ALLOCATION_DIR / f"{sample}_allocation.tsv") as h:
        for r in csv.DictReader(h, delimiter="\t"):
            n = int(r["simulated_read_pairs"])
            allocated_n += 1
            allocated_total += n
            allocated_nonzero += int(n > 0)

    pcr_summary.append({
        "sample": sample,
        "fragmentation_model": frag_summary["fragmentation_model"],
        "pcr1_templates": pcr1_n,
        "pcr1_molecules": pcr1_molecules,
        "library_input_molecules": int(frag_summary["library_input_molecules"]),
        "fragmentation_events": int(frag_summary["fragmentation_events"]),
        "terminal_fragment_molecules": int(frag_summary["terminal_fragment_molecules"]),
        "terminal_fragment_species": int(frag_summary["terminal_fragment_species"]),
        "retained_fragment_molecules": fragment_input_molecules,
        "retained_fragment_species": len(frag_rows),
        "discarded_fragment_molecules": int(frag_summary["discarded_fragment_molecules"]),
        "input_nt_mass": int(frag_summary["input_nt_mass"]),
        "terminal_nt_mass": int(frag_summary["terminal_nt_mass"]),
        "fragmentation_nt_conserved": True,
        "pcr2_fragments": len(copies2),
        "target_read_pairs": allocated_total,
        "fragments_with_reads": allocated_nonzero,
        "fragment_sampling_dropout": 1 - allocated_nonzero / allocated_n,
        "mean_pcr2_efficiency": float(np.mean(eff2)),
        "median_pcr2_copies": float(np.median(copies2)),
    })

out = QC_DIR / "pcr1_fragmentation_pcr2_summary.tsv"
write_dict_rows(out, pcr_summary, list(pcr_summary[0]))
print(*pcr_summary, sep="\n")


## 12a. Восстановимость эталонных шаблонов (reconstructability)

Для каждого эталонного шаблона по фрагментам, получившим риды, определяется, что сборщик **в принципе** может восстановить:

| класс | смысл |
|---|---|
| `no_reads` | шаблон не попал в FASTQ |
| `partial_read_coverage` | часть V…J не прочитана ни одним ридом |
| `full_coverage_unlinked` | все позиции прочитаны, но есть точка, которую не пересекает ни один фрагмент: куски физически не связаны, их нельзя соединить ни перекрытием, ни mate-парами |
| `full_coverage_mate_linked_only` | каждая точка пересечена фрагментом, но хотя бы одна — только несеквенированным промежутком между R1 и R2 (проверка mate-pair extension TRUST4) |
| `full_coverage_read_overlap` | каждая точка лежит внутри рида с `RECONSTRUCT_MIN_FLANK_NT` с обеих сторон — сборка обычным перекрытием |

Эти классы — знаменатели для benchmark TRUST4: recall считается отдельно по каждому классу.


In [ ]:
# Класс восстановимости эталонного интервала V…J каждого шаблона по тому, что реально попало в FASTQ.
# ISS amplicon: R1 = [start, start+R), R2 = [end-R, end); вся вставка — физическая связь.
# Связь (bond) b между позициями b-1 и b:
#   read_overlap — b лежит внутри одного рида с >= K нт с каждой стороны
#                  (собирается обычным перекрытием);
#   mate_only    — b внутри фрагмента, но в несеквенированном промежутке между
#                  R1 и R2 (соединить может только mate-pair extension TRUST4);
#   unlinked     — ни один фрагмент не пересекает b: куски шаблона физически
#                  не связаны, соединить их нельзя без внешней информации.
RECONSTRUCT_MIN_FLANK_NT = 31  # ponytail: грубый аналог k-mer/min-overlap сборщика, калибруемый

def _bond_mask(L, intervals):
    d = np.zeros(L + 2, dtype=np.int32)
    for a, b in intervals:
        a, b = max(a, 1), min(b, L - 1)
        if a <= b:
            d[a] += 1
            d[b + 1] -= 1
    return np.cumsum(d)[: L + 1] > 0

def classify_template(L, frags, R=TARGET_READ_LENGTH, K=RECONSTRUCT_MIN_FLANK_NT, region=None):
    """frags: [(start, end)] фрагментов с ридами в координатах шаблона.

    region=(vj_start, vj_end): оценивается только эталонный интервал V…J
    (праймеры и leader/C-фланги не требуют сборки). Возвращает (class, read_covered_fraction).
    """
    reads = [iv for s, e in frags for iv in ((s, min(s + R, e)), (max(e - R, s), e))]
    if region is not None:
        a0, b0 = region
        clip = lambda ivs: [(max(a, a0) - a0, min(b, b0) - a0) for a, b in ivs if min(b, b0) > max(a, a0)]
        L, frags, reads = b0 - a0, clip(frags), clip(reads)
    if not frags:
        return "no_reads", 0.0
    cov = np.zeros(L + 1, dtype=np.int32)
    for a, b in reads:
        cov[a] += 1
        cov[b] -= 1
    covered = float((np.cumsum(cov)[:L] > 0).mean())
    if covered < 1.0:
        return "partial_read_coverage", covered
    by_frag = _bond_mask(L, [(s + K, e - K) for s, e in frags])
    by_read = _bond_mask(L, [(a + K, b - K) for a, b in reads])
    bonds = slice(K, L - K + 1)  # связи ближе K к краям шаблона не требуют соединения
    if not by_frag[bonds].all():
        return "full_coverage_unlinked", covered
    if not by_read[bonds].all():
        return "full_coverage_mate_linked_only", covered
    return "full_coverage_read_overlap", covered

# self-check: 400 нт, PE150
assert classify_template(400, [])[0] == "no_reads"
assert classify_template(400, [(0, 200), (200, 400)])[0] == "full_coverage_unlinked"
assert classify_template(300, [(0, 200), (100, 300)])[0] == "full_coverage_read_overlap"
assert classify_template(400, [(0, 400)])[0] == "partial_read_coverage"          # середина между R1 и R2 не прочитана
assert classify_template(300, [(0, 300)])[0] == "full_coverage_mate_linked_only"  # R1 и R2 стыкуются без перекрытия
assert classify_template(420, [(0, 420), (100, 320)])[0] == "full_coverage_mate_linked_only"
# фланги вне V…J не учитываются; фрагмент только в праймере = нет ридов в V…J
assert classify_template(500, [(0, 300), (200, 500)], region=(50, 450))[0] == "full_coverage_mate_linked_only"
assert classify_template(500, [(0, 40)], region=(50, 450))[0] == "no_reads"

def write_reconstructability_qc(sample):
    frags = defaultdict(list)
    for r in read_dict_rows(ALLOCATION_DIR / f"{sample}_allocation.tsv"):
        if int(r["simulated_read_pairs"]) > 0:
            s = int(r["fragment_start"])
            frags[r["template_id"]].append((s, s + int(r["fragment_length"])))
    rows, summary = [], defaultdict(lambda: defaultdict(int))
    for t in load_dedup_templates(sample):
        cls, cov = classify_template(t["length"], frags.get(t["template_id"], []),
                                     region=(t["vj_start"], t["vj_end"]))
        rows.append({"template_id": t["template_id"], "locus": t["locus"], "length": t["length"],
                     "unique_umis": t["unique_umis"],
                     "fragments_with_reads": len(frags.get(t["template_id"], [])),
                     "read_covered_fraction": round(cov, 4), "reconstructability": cls})
        summary[t["locus"]][cls] += 1
    write_dict_rows(QC_DIR / f"{sample}_template_reconstructability.tsv", rows, list(rows[0]))
    classes = ["no_reads", "partial_read_coverage", "full_coverage_unlinked",
               "full_coverage_mate_linked_only", "full_coverage_read_overlap"]
    out = []
    for locus in sorted(summary) + ["ALL"]:
        c = {k: (sum(summary[l][k] for l in summary) if locus == "ALL" else summary[locus][k]) for k in classes}
        n = sum(c.values())
        out.append({"sample": sample, "locus": locus, "templates": n, **c,
                    **{f"{k}_frac": round(c[k] / n, 4) for k in classes}})
    write_dict_rows(QC_DIR / f"{sample}_reconstructability_summary.tsv", out, list(out[0]))
    print(*out, sep="\n")

for sample in SAMPLES:
    write_reconstructability_qc(sample)


## 13. Генерация ридов точной длины PE150


In [ ]:
def run_iss_generate(sample,force=FORCE):
    fa=ALLOCATION_DIR/f"{sample}_selected_fragments.fasta"; counts=ALLOCATION_DIR/f"{sample}_read_counts.tsv"; prefix=FASTQ_DIR/sample; ext=".fastq.gz"; r1=Path(str(prefix)+f"_R1{ext}"); r2=Path(str(prefix)+f"_R2{ext}")
    if r1.exists() and r2.exists() and not force: print("[skip]",sample); return
    cmd=["iss","generate","--genomes",str(fa),"--readcount_file",str(counts),"--sequence_type",SEQUENCE_TYPE,"--model",str(CUSTOM_MODEL_NPZ),"--cpus",str(NPROC),"--output",str(prefix),"--seed",str(sample_seed(sample,5000)),"--compress"]
    run_with_heartbeat(cmd,LOGS_DIR/f"{sample}_iss_custom.log"); n1=count_fastq(r1); n2=count_fastq(r2); assert n1==n2==READ_BUDGETS[sample]; print("custom empirical PE150",n1,"pairs")
for sample in SAMPLES: run_iss_generate(sample)


## 14. Итоговая проверка


In [ ]:
final_rows=[]
for sample in SAMPLES:
    r1=FASTQ_DIR/f"{sample}_R1.fastq.gz"; r2=FASTQ_DIR/f"{sample}_R2.fastq.gz"; n1=count_fastq(r1); n2=count_fastq(r2)
    lengths1=sorted({len(s) for s in iter_fastq(r1)}); lengths2=sorted({len(s) for s in iter_fastq(r2)})
    final_rows.append({"branch":BRANCH_NAME,"sample":sample,"expected_pairs":READ_BUDGETS[sample],"R1_reads":n1,"R2_reads":n2,"R1_lengths":",".join(map(str,lengths1)),"R2_lengths":",".join(map(str,lengths2)),"valid":n1==n2==READ_BUDGETS[sample] and lengths1==lengths2==[TARGET_READ_LENGTH]})
out=QC_DIR/"final_qc.tsv"; write_dict_rows(out,final_rows,list(final_rows[0])); assert all(r["valid"] for r in final_rows); print(*final_rows,sep="\n")


## Интерпретация и научные ограничения

- Биологический эталон ограничен репертуаром после фильтрации аннотаций.
- Шаблон — физический ампликон PCR1 (V-leader праймер + склеенный рид + C-праймер); эталон для оценки сборки — интервал `vj_start…vj_end`. UMI и Illumina-хвосты в шаблон не включены; вырожденные позиции праймеров разрешены случайно, но воспроизводимо.
- При `UMI_COLLAPSE_MODE=umi_mode` каждая UMI-молекула считается один раз; одиночные UMI с ошибкой секвенирования по-прежнему дают отдельный вариант.
- Модель ошибок независимо обучается по исходным AIRR-аннотациям и соответствующим парам `pr_trimmed`, поэтому последующая фильтрация не занижает оценку ошибок.
- Консенсус UMI-семейства сохраняет общие биологические варианты, включая SHM; germline IgBLAST используется только как независимая проверка правдоподобия.
- Семейства меньше трёх записей могут входить в биологический эталон, но не подходят для сравнения консенсуса с отложенным ридом.
- PCR1 и PCR2 моделируются стохастическим ветвлением. Замены полимеразы, химеры, истощение реагентов и изменение эффективности по циклам явно не моделируются.
- Смешанный IGH+IGK+IGL набор является вычислительным тестом без нативного спаривания тяжёлой и лёгкой цепей.
- Между PCR1 и fragmentation явно моделируется конечный library-input aliquot; до size-selection проверяется сохранение суммарной нуклеотидной массы.
- Фрагментация использует приближённую ultrasonic-like recursive mechanical-shearing модель; она не претендует на моделирование конкретного прибора.
